In [90]:
from extractor import extract_all_files
from transform import cast_types, fill_nulls, add_columns, filter_rows, joins
from loader import load_layer, validate_load, load_summary
from storage import save_csv

### CAMINHOS E CONFIGS

In [91]:
RAW_PATH       = r"C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\Raw"
PROCESSED_PATH = r"C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data"

In [92]:
FILES = {
    "deliveries": {
        "file":   "deliveries.csv",
        "schema": ["delivery_id", "driver_id", "hub_id", "status",
                   "shipped_date", "delivered_date", "distance_km",
                   "cost", "customer_rating"],
    },
    "drivers": {"file": "drivers.csv", "schema": ["driver_id", "name", "hub_id"]},
    "hubs":    {"file": "hubs.csv",    "schema": ["hub_id", "city", "state"]},
}

### EXTRACT

In [93]:
print("\n===EXTRACT===")
dfs = extract_all_files(RAW_PATH, FILES)


===EXTRACT===
[extract] deliveries.csv | 102000 linhas | 11 colunas
[extract] drivers.csv | 255 linhas | 3 colunas
[extract] hubs.csv | 10 linhas | 3 colunas


### TRANSFORM

In [94]:
print("\n===TRANSFORM===")
df = dfs["deliveries"].copy()


===TRANSFORM===


In [95]:
df.info()

df = cast_types(df, {
    "shipped_date":    "datetime",
    "delivered_date":  "datetime",
    "distance_km":     "float",
    "cost":            "float",
    "customer_rating": "float",
})

<class 'pandas.DataFrame'>
RangeIndex: 102000 entries, 0 to 101999
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   delivery_id      102000 non-null  int64  
 1   driver_id        102000 non-null  int64  
 2   hub_id           102000 non-null  int64  
 3   status           102000 non-null  str    
 4   shipped_date     102000 non-null  str    
 5   delivered_date   64717 non-null   str    
 6   distance_km      102000 non-null  str    
 7   cost             102000 non-null  float64
 8   customer_rating  96983 non-null   float64
 9   created_at       102000 non-null  str    
 10  updated_at       102000 non-null  str    
dtypes: float64(2), int64(3), str(6)
memory usage: 13.5 MB
[transform] 'shipped_date' -> datetime
[transform] 'delivered_date' -> datetime
[transform] 'distance_km' -> float
[transform] 'cost' -> float
[transform] 'customer_rating' -> float


In [96]:
df.head()

,delivery_id,driver_id,hub_id,status,shipped_date,delivered_date,distance_km,cost,customer_rating,created_at,updated_at
0,1,157,1,canceled,2026-02-05,NaT,103.39,549.91,1.1,2026-02-01,2026-02-05
1,2,387,4,canceled,2026-03-25,NaT,449.76,870.61,1.6,2026-03-24,2026-03-27
2,3,152,2,delayed,2026-01-13,2026-01-24,807.32,540.87,2.5,2026-01-09,2026-01-15
3,4,395,4,canceled,2026-01-09,NaT,290.10,240.46,2.1,2026-01-07,2026-01-11
4,5,207,5,canceled,2026-04-30,NaT,635.34,252.36,2.1,2026-04-29,2026-05-05


In [97]:
df["status"].unique()

<ArrowStringArray>
['canceled', 'delayed', 'delivered']
Length: 3, dtype: str

In [99]:
df = filter_rows(df, {"status": ["delayed", "delivered"]}, "deliveries")

df = fill_nulls(df, {"customer_rating": 0.0})
df = add_columns(df, {
    "delivery_days": lambda d: (d["delivered_date"] - d["shipped_date"]).dt.days,
    "cost_per_km":   lambda d: (d["cost"] / d["distance_km"]).round(2),
})

df["status"].unique()

[transform] 'deliveries' — filter {'status': ['delayed', 'delivered']}: 0 linhas removidas
[transform] 'customer_rating' - 0 nulos preenchidos com '0.0'
[transform] Coluna 'delivery_days' criada
[transform] Coluna 'cost_per_km' criada


<ArrowStringArray>
['delayed', 'delivered']
Length: 2, dtype: str

In [100]:
df_drivers         = dfs["drivers"].copy()
df_drivers["name"] = df_drivers["name"].str.strip().str.title()
df_hubs            = dfs["hubs"].copy()
df_hubs["city"]    = df_hubs["city"].str.strip().str.title()
df_hubs["state"]   = df_hubs["state"].str.strip().str.upper()


In [101]:
df = joins(df, df_drivers, on="driver_id", cols=["driver_id", "name"])

df = joins(df, df_hubs,    on="hub_id",    cols=["hub_id", "city", "state"])

df = df.rename(columns={"name": "driver_name"})


[transform] enrich on 'driver_id' (left) - shape: (68049, 14)
[transform] enrich on 'hub_id' (left) - shape: (68049, 16)


### LOAD RAW

In [102]:
print("\n===LOAD: RAW===")
for name, df_raw in dfs.items():
    load_layer(df_raw, PROCESSED_PATH, "raw", f"raw_{name}")


===LOAD: RAW===
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\raw\parquet\raw_deliveries.parquet' - 102000 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\raw\csv\raw_deliveries.csv' - 102000 linhas
[load] 'raw_deliveries' -> camada 'raw'
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\raw\parquet\raw_drivers.parquet' - 255 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\raw\csv\raw_drivers.csv' - 255 linhas
[load] 'raw_drivers' -> camada 'raw'
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\raw\parquet\raw_hubs.parquet' - 10 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\mont

### LOAD PROCESSED

In [85]:
print("\n===LOAD: PROCESSED===")

load_layer(df, PROCESSED_PATH, "processed", "deliveries_transformed")


===LOAD: PROCESSED===
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed\parquet\deliveries_transformed.parquet' - 68049 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed\csv\deliveries_transformed.csv' - 68049 linhas
[load] 'deliveries_transformed' -> camada 'processed'


### LOAD ANALYTICS

In [86]:
print("\n=== LOAD: ANALYTICS ===")

df_by_hub = df.groupby(["city", "state"]).agg(
    total_deliveries=("delivery_id",     "count"),
    total_revenue   =("cost",            "sum"),
    avg_rating      =("customer_rating", "mean"),
    avg_days        =("delivery_days",   "mean"),
).round(2).reset_index()

load_layer(df_by_hub, PROCESSED_PATH, "analytics", "deliveries_by_hub")


=== LOAD: ANALYTICS ===
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\analytics\parquet\deliveries_by_hub.parquet' - 10 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\analytics\csv\deliveries_by_hub.csv' - 10 linhas
[load] 'deliveries_by_hub' -> camada 'analytics'


In [87]:
df_by_driver = df.groupby(["driver_id", "driver_name"]).agg(
    total_deliveries=("delivery_id",     "count"),
    total_revenue   =("cost",            "sum"),
    avg_rating      =("customer_rating", "mean"),
    avg_days        =("delivery_days",   "mean"),
).round(2).reset_index()

load_layer(df_by_driver, PROCESSED_PATH, "analytics", "deliveries_by_driver")


[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\analytics\parquet\deliveries_by_driver.parquet' - 255 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\analytics\csv\deliveries_by_driver.csv' - 255 linhas
[load] 'deliveries_by_driver' -> camada 'analytics'


In [88]:
df_by_status = df.groupby("status").agg(
    total_deliveries=("delivery_id",  "count"),
    avg_cost        =("cost",         "mean"),
    avg_distance    =("distance_km",  "mean"),
).round(2).reset_index()

load_layer(df_by_status, PROCESSED_PATH, "analytics", "deliveries_by_status")

[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\analytics\parquet\deliveries_by_status.parquet' - 2 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\analytics\csv\deliveries_by_status.csv' - 2 linhas
[load] 'deliveries_by_status' -> camada 'analytics'


### VALIDATE

In [89]:
print("\n=== VALIDAR CARGA ===")
validate_load(
    f"{PROCESSED_PATH}\\deliveries_transformed.parquet",
    expected_rows=len(df),
    expected_cols=len(df.columns)
)


=== VALIDAR CARGA ===


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Ceifas\\OneDrive\\Desktop\\data-engineering-portfolio\\projects\\month02_Pipeline\\Data\\deliveries_transformed.parquet'

In [ ]:
print("\n===RESUMO===")
df_summary = load_summary({
    "raw": [
        f"{PROCESSED_PATH}\\raw_deliveries.parquet",
        f"{PROCESSED_PATH}\\raw_drivers.parquet",
        f"{PROCESSED_PATH}\\raw_hubs.parquet",
    ],
    "processed": [
        f"{PROCESSED_PATH}\\deliveries_transformed.parquet",
    ],
    "analytics": [
        f"{PROCESSED_PATH}\\analytics\\deliveries_by_hub.parquet",
        f"{PROCESSED_PATH}\\analytics\\deliveries_by_driver.parquet",
        f"{PROCESSED_PATH}\\analytics\\deliveries_by_status.parquet",
    ]
})


===RESUMO===


In [ ]:
print(df_summary.to_string(index=False))
save_csv(df_summary, f"{PROCESSED_PATH}\\analytics\\load_summary.csv")

   camada                        arquivo  linhas  colunas tamanho
      raw         raw_deliveries.parquet       0        0    0 KB
      raw            raw_drivers.parquet       0        0    0 KB
      raw               raw_hubs.parquet       0        0    0 KB
processed deliveries_transformed.parquet       0        0    0 KB
analytics      deliveries_by_hub.parquet       0        0    0 KB
analytics   deliveries_by_driver.parquet       0        0    0 KB
analytics   deliveries_by_status.parquet       0        0    0 KB
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\analytics\load_summary.csv' — 7 linhas
